In [1]:
import os
import json
from pathlib import Path
from collections import defaultdict

from google.cloud import vision
from google.cloud import storage
from tqdm import tqdm

# ---- GCP CONFIG ----
# Set your project ID if needed; if your env already has a default project, you can leave this as None.
PROJECT_ID = 'turing-agent-358210'  # e.g. 'your-gcp-project-id' or None to use default
BUCKET_NAME = 'capstone-ii-applicant-documents'

# Folder where PDFs live in the bucket
INPUT_PREFIX = 'applicant-documents-pdf/'

# Folder in the SAME bucket where Vision async outputs will be written
VISION_OUTPUT_PREFIX = 'vision-ocr-output'

# Folder in the bucket where final per-applicant JSON files will be stored
RESULTS_PREFIX = 'vision-ocr-results'

# Optional local directory to store per-applicant JSONs (for debugging)
#LOCAL_JSON_DIR = Path('ocr_json')
#LOCAL_JSON_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_JSON_DIR = None  # Set to None to skip local saving

# If you need to set credentials here (otherwise set them in env BEFORE starting Jupyter):
# os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = str(Path('path/to/service_account.json'))

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = "../../turing-agent-358210-a38a4820a9ce.json"

# ---- CLIENTS ----
storage_client = storage.Client(project=PROJECT_ID) if PROJECT_ID else storage.Client()
vision_client = vision.ImageAnnotatorClient()


In [2]:
def parse_gs_uri(gs_uri: str):
    """Parse a gs://bucket/path URI into (bucket_name, prefix).

    Ensures prefix ends with '/' if not empty.
    """
    if not gs_uri.startswith('gs://'):
        raise ValueError(f'Invalid GCS URI: {gs_uri}')
    no_scheme = gs_uri[5:]
    parts = no_scheme.split('/', 1)
    bucket = parts[0]
    prefix = parts[1] if len(parts) > 1 else ''
    if prefix and not prefix.endswith('/'):
        prefix += '/'
    return bucket, prefix


In [3]:
def list_applicant_pdfs(bucket_name: str, input_prefix: str):
    """Scan bucket and build a dict: { applicant_id: [blob, ...] }.

    Assumes structure:
        input_prefix/applicant_#/document_name.pdf
    """
    pdf_exts = {'.pdf'}
    applicant_docs = defaultdict(list)

    blobs_iter = storage_client.list_blobs(bucket_name, prefix=input_prefix)

    for blob in blobs_iter:
        # Skip 'folder' markers
        if blob.name.endswith('/'):
            continue

        ext = Path(blob.name).suffix.lower()
        if ext not in pdf_exts:
            continue

        # Path relative to input_prefix, e.g. 'applicant_123/doc.pdf'
        rel_path = blob.name[len(input_prefix):].lstrip('/')
        parts = rel_path.split('/')
        if len(parts) < 2:
            # Not in expected folder structure
            continue

        applicant_folder = parts[0]  # e.g. 'applicant_123'
        if '_' in applicant_folder:
            applicant_id = applicant_folder.split('_', 1)[1]
        else:
            applicant_id = applicant_folder

        applicant_docs[applicant_id].append(blob)

    return applicant_docs

applicant_docs = list_applicant_pdfs(BUCKET_NAME, INPUT_PREFIX)
print(f'Found {len(applicant_docs)} applicants with PDFs.')
print('Sample applicant IDs:', list(applicant_docs.keys())[:5])


Found 4 applicants with PDFs.
Sample applicant IDs: ['1', '2', '3', '4']


In [4]:
def ocr_pdf_gcs(gcs_source_uri: str, gcs_destination_uri: str, timeout: int = 600) -> str:
    """Run async OCR on a PDF in GCS and return combined text for all pages.

    Parameters
    ----------
    gcs_source_uri : str
        e.g. 'gs://capstone-ii-applicant-documents/applicant-documents-image/applicant_123/doc.pdf'
    gcs_destination_uri : str
        e.g. 'gs://capstone-ii-applicant-documents/vision-ocr-output/applicant_123/doc/'
    timeout : int
        Seconds to wait for the async operation.
    """
    mime_type = 'application/pdf'
    feature = vision.Feature(type_=vision.Feature.Type.DOCUMENT_TEXT_DETECTION)

    gcs_source = vision.GcsSource(uri=gcs_source_uri)
    input_config = vision.InputConfig(gcs_source=gcs_source, mime_type=mime_type)

    gcs_destination = vision.GcsDestination(uri=gcs_destination_uri)
    # batch_size: how many pages per JSON output; adjust if needed
    output_config = vision.OutputConfig(gcs_destination=gcs_destination, batch_size=10)

    async_request = vision.AsyncAnnotateFileRequest(
        features=[feature],
        input_config=input_config,
        output_config=output_config,
    )

    operation = vision_client.async_batch_annotate_files(requests=[async_request])
    response = operation.result(timeout=timeout)

    # Real OCR results are written to GCS; now read them back
    dest_bucket_name, dest_prefix = parse_gs_uri(gcs_destination_uri)
    blobs = list(storage_client.list_blobs(dest_bucket_name, prefix=dest_prefix))
    if not blobs:
        raise RuntimeError(f'No OCR output files found at {gcs_destination_uri}')

    full_text = []

    for blob in blobs:
        json_str = blob.download_as_text(encoding='utf-8')
        result = json.loads(json_str)

        for resp in result.get('responses', []):
            annotation = resp.get('fullTextAnnotation')
            if annotation and 'text' in annotation:
                full_text.append(annotation['text'])

    return '\n'.join(full_text)


In [5]:
def upload_json_to_gcs(bucket_name: str, blob_name: str, data: dict):
    """Upload a JSON-serializable dict to GCS as a .json file."""
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(blob_name)
    json_str = json.dumps(data, ensure_ascii=False, indent=2)
    blob.upload_from_string(json_str, content_type='application/json')
    print(f'Uploaded JSON to gs://{bucket_name}/{blob_name}')


In [6]:
def build_applicant_jsons(
    applicant_docs: dict,
    bucket_name: str,
    input_prefix: str,
    vision_output_prefix: str,
    results_prefix: str,
    local_json_dir: Path | None = None,
    max_applicants: int | None = None,
):
    """Iterate over applicants, OCR their PDFs, and write one JSON per applicant.

    Outputs:
      - Uploads per-applicant JSON into:
          gs://<bucket>/<results_prefix>/applicant_<id>/applicant_<id>.json
      - Optionally writes a local JSON copy if local_json_dir is provided.
    """
    applicant_ids = sorted(applicant_docs.keys())
    if max_applicants is not None:
        applicant_ids = applicant_ids[:max_applicants]

    for applicant_id in tqdm(applicant_ids, desc='Processing applicants'):
        blobs = applicant_docs[applicant_id]
        applicant_key = f'applicant_{applicant_id}'

        result = {
            'applicant_id': applicant_id,
            'documents': {},
        }

        for blob in blobs:
            # e.g. 'applicant-documents-image/applicant_123/passport.pdf'
            doc_basename = Path(blob.name).stem  # 'passport'
            gcs_source_uri = f'gs://{bucket_name}/{blob.name}'

            # Destination folder for Vision outputs
            dest_prefix = f'{vision_output_prefix}/{applicant_key}/{doc_basename}/'
            gcs_destination_uri = f'gs://{bucket_name}/{dest_prefix}'

            try:
                text = ocr_pdf_gcs(gcs_source_uri, gcs_destination_uri)
            except Exception as e:
                print(f'Error OCR\'ing {gcs_source_uri}: {e}')
                text = ''

            result['documents'][doc_basename] = text

        # Upload per-applicant JSON to GCS
        gcs_result_blob = f'{results_prefix}/{applicant_key}/{applicant_key}.json'
        upload_json_to_gcs(bucket_name=bucket_name, blob_name=gcs_result_blob, data=result)

        # Optional: also write locally
        if local_json_dir is not None:
            local_json_dir.mkdir(parents=True, exist_ok=True)
            json_path = local_json_dir / f'{applicant_key}.json'
            with open(json_path, 'w', encoding='utf-8') as f:
                json.dump(result, f, ensure_ascii=False, indent=2)
            print(f'Saved local JSON for {applicant_key} to {json_path}')


In [7]:
# Run the full pipeline
# For a small test, you can pass max_applicants=1 or 2.
build_applicant_jsons(
    applicant_docs=applicant_docs,
    bucket_name=BUCKET_NAME,
    input_prefix=INPUT_PREFIX,
    vision_output_prefix=VISION_OUTPUT_PREFIX,
    results_prefix=RESULTS_PREFIX,
    local_json_dir=LOCAL_JSON_DIR,  # or None to skip local copies
    # max_applicants=2,
)


Processing applicants:  25%|██▌       | 1/4 [09:16<27:50, 556.88s/it]

Uploaded JSON to gs://capstone-ii-applicant-documents/vision-ocr-results/applicant_1/applicant_1.json


Processing applicants:  50%|█████     | 2/4 [14:59<14:21, 430.96s/it]

Uploaded JSON to gs://capstone-ii-applicant-documents/vision-ocr-results/applicant_2/applicant_2.json


Processing applicants:  75%|███████▌  | 3/4 [21:42<06:58, 418.08s/it]

Uploaded JSON to gs://capstone-ii-applicant-documents/vision-ocr-results/applicant_3/applicant_3.json


Processing applicants: 100%|██████████| 4/4 [23:23<00:00, 350.86s/it]

Uploaded JSON to gs://capstone-ii-applicant-documents/vision-ocr-results/applicant_4/applicant_4.json
